# ULTRA Fine-Tuning on MIND splits_updated

Fine-tunes the pretrained ULTRA checkpoint (`ultra_50g`) on each fold of MIND `splits_updated`.  
Zero-shot ULTRA already achieves MRR = 0.357; fine-tuning targets MRR > 0.45.

**Strategy:** Load `ultra_50g` weights, freeze the relation encoder, fine-tune the entity encoder  
for a small number of epochs on each fold's training indication edges, then evaluate on test/valid.

**Output:** `finetuned_predictions_test.tsv` and `finetuned_predictions_valid.tsv` per slice.

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────
import subprocess, sys, os

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
    else:
        print(result.stdout[-200:] or 'OK')
    return result

# PyTorch 2.2.2+cu118 supports sm_60 (P100), sm_75 (T4), sm_80 (A100)
print('Installing PyTorch 2.2.2+cu118...')
run('pip install -q torch==2.2.2+cu118 torchvision torchaudio '
    '--index-url https://download.pytorch.org/whl/cu118')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print(f'GPU: {torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}')

pyg_url = 'https://data.pyg.org/whl/torch-2.2.2+cu118.html'
run('pip install -q torch_geometric')
run(f'pip install -q torch_scatter torch_sparse torch_cluster -f {pyg_url}')

# Clone ULTRA
if not os.path.exists('/kaggle/working/ULTRA'):
    run('git clone --quiet https://github.com/DeepGraphLearning/ULTRA.git /kaggle/working/ULTRA')
else:
    print('ULTRA already cloned.')

sys.path.insert(0, '/kaggle/working/ULTRA')
print('Done.')

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import os, torch, numpy as np, pandas as pd
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
from torch import optim
from torch_geometric.data import Data

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
SLICES_TO_RUN  = list(range(5))
TOP_K          = 50
BATCH_SIZE     = 32      # smaller than zero-shot — gradients need memory
NUM_EPOCHS     = 15      # fine-tune epochs per slice
LR             = 1e-4    # learning rate
NEG_SAMPLES    = 32      # negative samples per positive triple
ADVERSARIAL_T  = 0.5     # self-adversarial temperature
MARGIN         = 6.0     # margin for training loss

# Find splits_updated
BASE_INPUT = Path('/kaggle/input')
splits_root = None
for p in BASE_INPUT.rglob('slice_0'):
    if (p / 'ind_test.tsv').exists():
        splits_root = p.parent
        break

assert splits_root is not None, 'Could not find splits_updated dataset'
print(f'Splits: {splits_root}')

OUT_BASE = Path('/kaggle/working/predictions/ULTRA_finetuned')
CKPT_BASE = Path('/kaggle/working/checkpoints')
OUT_BASE.mkdir(parents=True, exist_ok=True)
CKPT_BASE.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Load pretrained ULTRA checkpoint ──────────────────────────────────────
from ultra.models import Ultra as UltraModel

ckpt_path = '/kaggle/working/ULTRA/ckpts/ultra_50g.pth'
assert os.path.exists(ckpt_path), f'Checkpoint not found: {ckpt_path}'

rel_cfg = dict(
    **{'class': 'RelNBFNet'},
    input_dim=64, hidden_dims=[64,64,64,64,64,64],
    message_func='distmult', aggregate_func='sum',
    short_cut=True, layer_norm=True,
)
entity_cfg = dict(
    **{'class': 'EntityNBFNet'},
    input_dim=64, hidden_dims=[64,64,64,64,64,64],
    message_func='distmult', aggregate_func='sum',
    short_cut=True, layer_norm=True,
)

def fresh_model():
    m = UltraModel(rel_model_cfg=rel_cfg, entity_model_cfg=entity_cfg)
    state = torch.load(ckpt_path, map_location='cpu')
    m.load_state_dict(state['model'])
    return m.to(DEVICE)

print('Pretrained checkpoint ready.')
print(f'Parameters: {sum(p.numel() for p in fresh_model().parameters()):,}')

In [ ]:
# ── Graph + data helpers ───────────────────────────────────────────────────
from ultra.tasks import build_relation_graph

def load_triples(path):
    df = pd.read_csv(path, sep='\t', header=None, names=['h','r','t'])
    return list(zip(df['h'], df['r'], df['t']))

def build_graph(triples, entity2id, rel2id):
    src, dst, edge_type = [], [], []
    num_rels = len(rel2id)
    for h, r, t in triples:
        hid, tid, rid = entity2id[h], entity2id[t], rel2id[r]
        src.append(hid);  dst.append(tid);  edge_type.append(rid)
        src.append(tid);  dst.append(hid);  edge_type.append(rid + num_rels)
    data = Data(
        edge_index=torch.tensor([src, dst], dtype=torch.long),
        edge_type=torch.tensor(edge_type, dtype=torch.long),
        num_nodes=len(entity2id),
        num_relations=num_rels * 2,
    )
    return build_relation_graph(data)

print('Helpers defined.')

In [ ]:
# ── Training: self-adversarial negative sampling loss ─────────────────────

def sample_negatives(pos_h, pos_t, all_disease_ids, n_neg=NEG_SAMPLES):
    """Corrupt the tail (disease) to generate negatives."""
    neg_ids = np.random.choice(all_disease_ids, size=n_neg, replace=False)
    return torch.tensor(neg_ids, dtype=torch.long, device=DEVICE)


def adversarial_loss(pos_score, neg_scores, margin=MARGIN, temperature=ADVERSARIAL_T):
    """
    Self-adversarial negative sampling loss (Sun et al. RotatE).
    pos_score: scalar
    neg_scores: (n_neg,) tensor
    """
    # Adversarial weights
    weights = torch.softmax(neg_scores * temperature, dim=0).detach()
    neg_loss = (weights * torch.nn.functional.logsigmoid(neg_scores - margin)).sum()
    pos_loss = torch.nn.functional.logsigmoid(margin - pos_score)
    return -(pos_loss + neg_loss)


@torch.no_grad()
def evaluate(model, graph, queries, all_disease_ids, known_pairs, batch_size=BATCH_SIZE):
    """Run filtered evaluation; return dict of metrics."""
    model.eval()
    disease_id_list  = [d_id  for d_id, _  in all_disease_ids]
    disease_str_list = [d_str for _,  d_str in all_disease_ids]
    rows = []
    graph = graph.to(DEVICE)

    for i in tqdm(range(0, len(queries), batch_size), desc='Eval', leave=False):
        for q in queries[i:i+batch_size]:
            drug_str, exp_str = q['drug_str'], q['exp_str']
            h_id, r_id = q['h_id'], q['r_id']

            batch_tensor = torch.tensor(
                [[[h_id, d_id, r_id] for d_id in disease_id_list]],
                dtype=torch.long, device=DEVICE
            )
            scores = model(graph, batch_tensor)[0].cpu().float()

            disease_scores = {}
            for d_str, s in zip(disease_str_list, scores.tolist()):
                if (drug_str, d_str) in known_pairs and d_str != exp_str:
                    continue
                disease_scores[d_str] = s

            ranked = sorted(disease_scores, key=disease_scores.get, reverse=True)
            rank   = ranked.index(exp_str) + 1 if exp_str in ranked else len(ranked) + 1

            rows.append({
                'drug': drug_str, 'expected_disease': exp_str,
                'rank': rank, 'reciprocal_rank': 1.0 / rank,
                **{f'top{k+1}_disease': ranked[k] if k < len(ranked) else ''
                   for k in range(TOP_K)}
            })

    df = pd.DataFrame(rows)
    ranks = np.array(df['rank'])
    return df, {
        'MRR':     round(float(np.mean(1.0/ranks)), 4),
        'Hits@1':  round(float((ranks<=1).mean()), 4),
        'Hits@3':  round(float((ranks<=3).mean()), 4),
        'Hits@5':  round(float((ranks<=5).mean()), 4),
        'Hits@10': round(float((ranks<=10).mean()), 4),
    }

print('Training + eval helpers defined.')

In [ ]:
# ── Main fine-tuning loop ──────────────────────────────────────────────────
all_results = {}

for sl in SLICES_TO_RUN:
    print(f'\n{"="*55}')
    print(f'ULTRA fine-tuning — slice_{sl}')
    print(f'{"="*55}')

    slice_dir = splits_root / f'slice_{sl}'
    out_dir   = OUT_BASE / f'slice_{sl}'
    out_dir.mkdir(parents=True, exist_ok=True)

    # Build vocab from KGE training graph
    train_triples = load_triples(slice_dir / 'kge_train.tsv')
    entities  = sorted(set(h for h,r,t in train_triples) | set(t for h,r,t in train_triples))
    relations = sorted(set(r for h,r,t in train_triples))
    entity2id = {e: i for i, e in enumerate(entities)}
    rel2id    = {r: i for i, r in enumerate(relations)}
    print(f'  Entities: {len(entity2id):,}   Relations: {len(rel2id)}')

    ind_rel_id = rel2id.get('indication', 0)

    # Disease candidate set
    ind_triples = load_triples(slice_dir / 'ind_train.tsv')
    disease_strs = sorted(set(t for h, r, t in ind_triples))
    all_disease_ids = [(entity2id[d], d) for d in disease_strs if d in entity2id]
    disease_id_arr  = np.array([d_id for d_id, _ in all_disease_ids])
    print(f'  Disease candidates: {len(all_disease_ids):,}')

    # Build KG graph
    graph = build_graph(train_triples, entity2id, rel2id).to(DEVICE)
    print(f'  Graph edges: {graph.edge_index.shape[1]:,}')

    # Known pairs for filtered eval
    known_pairs = set()
    for split in ['train', 'test', 'valid']:
        for h, r, t in load_triples(slice_dir / f'ind_{split}.tsv'):
            known_pairs.add((h, t))

    # Fine-tuning triples (indication edges only)
    ft_triples = [(h, t) for h, r, t in ind_triples
                  if h in entity2id and t in entity2id]
    print(f'  Fine-tuning triples: {len(ft_triples)}')

    # Fresh model from pretrained checkpoint
    model = fresh_model()

    # Only fine-tune entity encoder; relation encoder stays frozen
    for name, param in model.named_parameters():
        if 'relation_model' in name:
            param.requires_grad = False
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Trainable params: {n_trainable:,} (entity encoder only)')

    optimizer = optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=LR
    )

    # Build eval queries
    def build_queries(split):
        df = pd.read_csv(slice_dir / f'ind_{split}.tsv',
                         sep='\t', header=None, names=['h','r','t'])
        qs, skipped = [], 0
        for _, row in df.iterrows():
            if row['h'] not in entity2id or row['t'] not in entity2id:
                skipped += 1; continue
            qs.append({'h_id': entity2id[row['h']], 'r_id': ind_rel_id,
                       't_id': entity2id[row['t']], 'drug_str': row['h'],
                       'exp_str': row['t']})
        print(f'    {split}: {len(qs)} queries ({skipped} skipped)')
        return qs

    valid_queries = build_queries('valid')
    test_queries  = build_queries('test')

    # ── Fine-tuning epochs ────────────────────────────────────────────────
    best_valid_mrr = 0.0
    best_ckpt_path = CKPT_BASE / f'ultra_ft_slice{sl}_best.pt'

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        np.random.shuffle(ft_triples)
        epoch_loss = 0.0

        for i in tqdm(range(0, len(ft_triples), BATCH_SIZE),
                      desc=f'Epoch {epoch}/{NUM_EPOCHS}', leave=False):
            batch = ft_triples[i:i+BATCH_SIZE]
            optimizer.zero_grad()
            batch_loss = torch.tensor(0.0, device=DEVICE, requires_grad=True)

            for drug, disease in batch:
                h_id = entity2id[drug]
                t_id = entity2id[disease]

                # Positive score
                pos_tensor = torch.tensor(
                    [[[h_id, t_id, ind_rel_id]]], dtype=torch.long, device=DEVICE
                )
                pos_score = model(graph, pos_tensor)[0, 0]

                # Negative scores (corrupt tail)
                neg_ids = np.random.choice(disease_id_arr, size=NEG_SAMPLES, replace=False)
                neg_tensor = torch.tensor(
                    [[[h_id, int(d_id), ind_rel_id] for d_id in neg_ids]],
                    dtype=torch.long, device=DEVICE
                )
                neg_scores = model(graph, neg_tensor)[0]

                loss = adversarial_loss(pos_score, neg_scores)
                batch_loss = batch_loss + loss

            batch_loss = batch_loss / len(batch)
            batch_loss.backward()
            optimizer.step()
            epoch_loss += batch_loss.item()

        # Validate
        _, valid_metrics = evaluate(model, graph, valid_queries,
                                    all_disease_ids, known_pairs)
        mrr = valid_metrics['MRR']
        print(f'  Epoch {epoch:02d}  loss={epoch_loss/max(1,len(ft_triples)//BATCH_SIZE):.4f}  '
              f'valid MRR={mrr:.4f}  H@10={valid_metrics["Hits@10"]:.4f}')

        if mrr > best_valid_mrr:
            best_valid_mrr = mrr
            torch.save({'model': model.state_dict(), 'epoch': epoch, 'mrr': mrr},
                       best_ckpt_path)
            print(f'    ** New best checkpoint saved (valid MRR={mrr:.4f})')

    # ── Final evaluation with best checkpoint ─────────────────────────────
    print(f'\n  Loading best checkpoint (valid MRR={best_valid_mrr:.4f})...')
    state = torch.load(best_ckpt_path, map_location=DEVICE)
    model.load_state_dict(state['model'])

    for split, queries in [('test', test_queries), ('valid', valid_queries)]:
        df_out, metrics = evaluate(model, graph, queries, all_disease_ids, known_pairs)
        all_results[(sl, split)] = metrics
        print(f'  {split}: MRR={metrics["MRR"]}  H@1={metrics["Hits@1"]}  '
              f'H@3={metrics["Hits@3"]}  H@5={metrics["Hits@5"]}  H@10={metrics["Hits@10"]}')
        df_out.to_csv(out_dir / f'finetuned_predictions_{split}.tsv', sep='\t', index=False)

print('\nAll slices done.')

In [ ]:
# ── Summary ────────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('ULTRA Fine-Tuned vs Zero-Shot — Summary')
print('='*60)

ZEROSHOT = {
    'MRR':     0.3569, 'Hits@1': 0.2824,
    'Hits@3':  0.3779, 'Hits@5': 0.4232, 'Hits@10': 0.5004,
}

for split in ['test', 'valid']:
    vals = {m: [all_results[(sl, split)][m]
                for sl in range(5) if (sl, split) in all_results]
            for m in ['MRR','Hits@1','Hits@3','Hits@5','Hits@10']}
    if not vals['MRR']:
        continue
    print(f'\n{split.upper()} SET (fine-tuned):')
    for m in ['MRR','Hits@1','Hits@3','Hits@5','Hits@10']:
        mean = np.mean(vals[m])
        std  = np.std(vals[m])
        zs   = ZEROSHOT.get(m, 0.0) if split == 'test' else 0.0
        gain = f'  (+{(mean-zs)/zs*100:.1f}% vs zero-shot)' if zs > 0 else ''
        print(f'  {m:<10}: {mean:.4f} +/- {std:.4f}{gain}')

print('\nPer-slice:')
for sl in range(5):
    for split in ['test', 'valid']:
        if (sl, split) in all_results:
            m = all_results[(sl, split)]
            print(f'  slice_{sl} {split}: MRR={m["MRR"]}  H@10={m["Hits@10"]}')

In [ ]:
# ── Zip output for download ────────────────────────────────────────────────
import shutil
shutil.make_archive('/kaggle/working/ultra_finetuned_predictions', 'zip',
                    '/kaggle/working/predictions')
print('Zipped: /kaggle/working/ultra_finetuned_predictions.zip')